# Project: Wildfire Mapping

Goal: Build an html side with an interactive map where the user can see the recents wildfires. Provide the user with information of phisical size, duration, intensity, etc. of the wildfires. Use pop-ups and tooltips to make the map interactive and structured for the users. The map should be for public users which are interesting in wildfires.

Tasks:
1. Load the api
2. Extract the data which is used to locate the wildfire (VIIRS_SNPP_NRT)
3. Explore the data
4. Clean the data (if needed)
5. Check wildfires for different properties
6. Visulaization of the wildfires
7. Provide additional information about the wildfires. 
8. Make the map interactive

In [12]:
# import all libraries
import requests
import pandas as pd
import io
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
from datetime import datetime

In [2]:
# Building the api url 

# get api key
api_key = "47778cf594ae276d2b7dfc098596de2a"
# define source
api_source = "VIIRS_SNPP_NRT" # or change ot to VIIRS_SNPP_SP?
# define area coordinates
api_area_coordinates = "world"
# day range. Days going back from today
api_day_range = 5
# build api url with api key
api_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{api_key}/{api_source}/{api_area_coordinates}/{api_day_range}"

In [3]:
# load the data form the api
response = requests.get(api_url)

# check if api import was successfull
if response.status_code == 200:
    print("API request successfull")

    # get the data as a csv
    data_csv = response.text
    # create a dataframe
    data_df = pd.read_csv(io.StringIO(data_csv))

else:
    print(f"Request failed. Status code: {response.status_code}")

API request successfull


In [19]:
# converting the data data frame into a geo-data frame
wildfire_gdf = gpd.GeoDataFrame(data_df, geometry=gpd.points_from_xy(data_df["longitude"], data_df["latitude"]))

# convert the date column into a date datetime type
wildfire_gdf["acq_date"] = pd.to_datetime(wildfire_gdf["acq_date"], format="%Y-%m-%d")

# verify transformation to geo data frame
display(wildfire_gdf.head(5))
display(wildfire_gdf.info)
display(wildfire_gdf.dtypes)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,geometry
0,54.84750,56.10764,295.84,0.56,0.69,2026-05-11,7,N,VIIRS,n,2.0NRT,271.35,1.34,N,POINT (56.10764 54.8475)
1,55.56430,51.93401,295.12,0.39,0.59,2026-05-11,7,N,VIIRS,n,2.0NRT,270.48,0.51,N,POINT (51.93401 55.5643)
2,55.59990,51.90491,296.59,0.39,0.59,2026-05-11,7,N,VIIRS,n,2.0NRT,274.84,0.63,N,POINT (51.90491 55.5999)
3,55.60163,51.89975,295.72,0.39,0.59,2026-05-11,7,N,VIIRS,n,2.0NRT,274.97,0.63,N,POINT (51.89975 55.60163)
4,55.61058,51.93056,295.52,0.39,0.59,2026-05-11,7,N,VIIRS,n,2.0NRT,274.78,1.00,N,POINT (51.93056 55.61058)


<bound method DataFrame.info of         latitude  longitude  bright_ti4  scan  track   acq_date  acq_time  \
0       54.84750   56.10764      295.84  0.56   0.69 2026-05-11         7   
1       55.56430   51.93401      295.12  0.39   0.59 2026-05-11         7   
2       55.59990   51.90491      296.59  0.39   0.59 2026-05-11         7   
3       55.60163   51.89975      295.72  0.39   0.59 2026-05-11         7   
4       55.61058   51.93056      295.52  0.39   0.59 2026-05-11         7   
...          ...        ...         ...   ...    ...        ...       ...   
139511  48.39699   36.72532      328.72  0.39   0.36 2026-05-15      1024   
139512  48.50945   37.72506      340.44  0.38   0.36 2026-05-15      1024   
139513  48.61396   35.23528      332.51  0.40   0.37 2026-05-15      1024   
139514  48.78996   44.82792      329.90  0.39   0.44 2026-05-15      1024   
139515  50.81564   45.21298      330.71  0.42   0.45 2026-05-15      1024   

       satellite instrument confidence vers

latitude             float64
longitude            float64
bright_ti4           float64
scan                 float64
track                float64
acq_date      datetime64[us]
acq_time               int64
satellite                str
instrument               str
confidence               str
version                  str
bright_ti5           float64
frp                  float64
daynight                 str
geometry            geometry
dtype: object

In [ ]:
# initalize Folium back ground map
background_map = folium.Map(
    location=[0, 0], # start zoom at latitude and longitude 0
    zoom_start=2, # shows the whole word at the start
    tiles="CartoDB DarkMatter" # Dark basmap
)

cluster_fire = MarkerCluster(name="Recent Fires").add_to(background_map)
# build markers for the wildfires
for idx, row in wildfire_gdf.iterrows():
    lat = row.geometry.y # extract latitude out of the geometry column 
    lon = row.geometry.x # eextract longitude out of the geometry coluumn

    # Formating the Tooltip. Days since the fire started
    date_today = datetime.today().date()
    fire_start = row["acq_date"].date()
    days_since = (date_today - fire_start).days
    days_of_fire = f"Days since fire start: {days_since}"

    # create Marker of the fire locations
    folium.Marker(
        location=[lat, lon],
        tooltip=days_of_fire,
        icon=folium.Icon(color="orange", icon="fire", prefix="fa")
    ).add_to(cluster_fire)

folium.LayerControl().add_to(background_map)
background_map
background_map.save("map.html")

datetime.date(2026, 5, 15)